# Measuring entity duplication in an LLM-built knowledge graph

This notebook runs one experiment end to end:

1. Build a corpus of documents whose entities are **known in advance**
2. Confirm a perfect extractor scores zero (the control)
3. Hand the same text to [Cognee](https://github.com/topoteretes/cognee) and count how many nodes it creates per real entity
4. Look at what actually got split
5. Repair the graph with `graphfaker.resolve()` and re-measure
6. Export the cleaned graph to Cypher / Neo4j

## What this measures

How often a pipeline emits **more than one node for one entity**, on clean input.

## What it does not measure

Answer quality, retrieval quality, or behaviour on messy real-world documents.
Those are different claims and this notebook does not support them.

Nothing here is corrupted. The documents are clean, well-formed English and
every entity is unambiguous to a human reader, so a correct pipeline scores
**zero**. Entities are referred to by the surface forms a normal writer uses —
full name, surname alone, an accepted abbreviation — which is ordinary prose,
not injected noise.

That restraint is deliberate. Synthetic *corruption* is far easier than
real-world error — Lam et al. ([IJPDS 2024](https://discovery.ucl.ac.uk/id/eprint/10194216/1/ijpds-09-2389.pdf))
measured real linkage error at 4.59% against 0.12% under naive corruption, a
~100x gap — so a benchmark built on guessed error rates mostly measures its own
noise model. Counting splits of entities a human would never split is a weaker
claim, and one a generator can actually support.

---

## Requirements

- `pip install graphfaker`
- For step 3: `pip install cognee` **and an LLM API key**. Cognee calls an LLM
  for every document, so this costs money. Start small (20 entities, 20
  documents) before scaling up.

Steps 1, 2, 4 and 6 run without cognee or a key.

In [ ]:
import json
import os
import sys

import networkx as nx

import graphfaker
from graphfaker import GraphFaker
from graphfaker.corpus import duplication_report, generate_corpus
from graphfaker.export import export_cypher, export_neo4j_csv
from graphfaker.resolve import resolve_entities

print("graphfaker", graphfaker.__version__)
print("networkx  ", nx.__version__)
print("python    ", sys.version.split()[0])

---
## Step 1 — Build a corpus with a known answer key

`generate_corpus` returns documents plus a registry of exactly which entities
they describe. Keep `SEED` fixed: a measurement made against an unreproducible
corpus cannot be checked by anyone else, including you next week.

Start small. `ENTITIES = 20, DOCUMENTS = 20` is enough to see the effect and
cheap to run; scale up once the pipeline works.

In [ ]:
SEED = 42
ENTITIES = 20
DOCUMENTS = 20

corpus = generate_corpus(seed=SEED, n_entities=ENTITIES, n_documents=DOCUMENTS)

print(f"{corpus.expected_entity_count} entities")
print(f"{len(corpus.documents)} documents")
print(f"{len(corpus.relations)} stated relations")

### Audit the corpus before trusting it

This is the step that makes the result defensible. If any surface form could be
claimed by two entities, then some of the "duplication" you measure is the
**corpus's** fault, not the pipeline's.

Two ways that happens, both of which bit me while building this:

- **Confusable names.** Naming four people `Person 0`..`Person 3` produces
  strings that are genuinely ~93% similar, so any name-based matcher merges
  them.
- **Alias collisions.** Faker derives place names from surnames, so a person
  aliased `Henderson` can appear beside an organization named
  `Henderson, Ramirez and Lewis`. Both have a real claim on the string.

`audit()` checks for both. The assertion below is not decoration — do not
publish a number from a corpus that fails it.

In [ ]:
audit = corpus.audit()

print(f"surface forms      : {audit['surface_forms']}")
print(f"ambiguous forms    : {len(audit['ambiguous_forms'])}")
print(f"containment pairs  : {len(audit['containment_pairs'])}")
print(f"clean              : {audit['clean']}")

if not audit["clean"]:
    print("\nambiguous:", audit["ambiguous_forms"])
    print("containment:", audit["containment_pairs"][:10])

assert audit["clean"], "Corpus is ambiguous - try a different SEED before measuring anything"
print("\nOK: no surface form is claimable by two entities.")

### What the text and the answer key look like

In [ ]:
document = corpus.documents[0]
print("DOCUMENT", document.id)
print(document.text)
print()
print("ENTITIES IT MENTIONS")
lookup = {e.id: e for e in corpus.entities}
for entity_id in document.entity_ids:
    entity = lookup[entity_id]
    print(f"  {entity.type:14s} {entity.name!r:34s} also written as {entity.aliases}")

In [ ]:
# Write the corpus to disk so any framework can read it, including ones that
# only accept a directory of files.
CORPUS_DIR = "corpus"
corpus.write(CORPUS_DIR)
print(f"{len(os.listdir(CORPUS_DIR))} files in {CORPUS_DIR}/ (documents + gold.json)")

---
## Step 2 — The control

A perfect extractor, one node per entity. **It must score zero.** If it does
not, the harness or the corpus is broken and every other number in this
notebook is meaningless.

Always run a control. It is the cheapest possible guard against measuring your
own bug and publishing it.

In [ ]:
def reference_extraction(corpus):
    G = nx.DiGraph()
    for entity in corpus.entities:
        G.add_node(entity.id, name=entity.name, type=entity.type)
    for relation in corpus.relations:
        G.add_edge(relation.source, relation.target, relationship=relation.type)
    return G


control = duplication_report(reference_extraction(corpus), corpus, framework="control")
print(control.summary())

assert control.duplication_rate == 0.0, "control must score zero"
assert control.missed == [], "control must find every entity"
print("\nControl passes.")

---
## Step 3 — Run Cognee

Verified against **cognee 1.4.1**: `add`, `cognify`, and `export` are all
coroutines, and `export` accepts `format="graphml"`. (There is no
`get_graph_data` function — that was a guess in an earlier draft of this
harness, and it was wrong.)

Two deliberate choices in the adapter:

- **It writes to a run-specific dataset instead of calling `cognee.prune`.**
  Pruning would wipe your local cognee store. `export` is scoped to a dataset,
  so isolation is achieved without destroying anything.
- **`data_cache=False`,** so a re-run genuinely re-extracts. Left on, a second
  run can replay the first one's output and look falsely stable.

> **This costs money and takes minutes.** Cognee calls an LLM per document.

In [ ]:
# Cognee reads the key from its own config; LLM_API_KEY is the usual variable.
# Set it in your shell rather than pasting a key into a notebook you might share.
HAVE_KEY = bool(
    os.environ.get("LLM_API_KEY")
    or os.environ.get("OPENAI_API_KEY")
    or os.environ.get("ANTHROPIC_API_KEY")
)

try:
    import cognee

    HAVE_COGNEE = True
    print("cognee", cognee.get_cognee_version())
except ImportError:
    HAVE_COGNEE = False
    print("cognee not installed - pip install cognee")

print("API key present:", HAVE_KEY)
if HAVE_COGNEE and not HAVE_KEY:
    # cognee's add() runs a pipeline that tests the LLM connection before
    # ingesting anything, so a missing key fails immediately rather than
    # part-way through.
    print("\nCognee needs a key even for add() - step 3 will be skipped.")

In [ ]:
import asyncio


async def _run_cognee(corpus, dataset):
    # One add() call with the whole list; cognee batches internally and
    # per-document calls are markedly slower.
    await cognee.add([d.text for d in corpus.documents], dataset_name=dataset)
    await cognee.cognify(datasets=[dataset], data_cache=False)
    destination = os.path.abspath(f"{dataset}.graphml")
    await cognee.export(dataset=dataset, format="graphml", destination=destination)
    return nx.read_graphml(destination)


cognee_raw = None
if HAVE_COGNEE and HAVE_KEY:
    DATASET = f"graphfaker_dup_{SEED}"
    cognee_raw = await _run_cognee(corpus, DATASET)   # notebooks allow top-level await
    print(f"cognee produced {cognee_raw.number_of_nodes()} nodes, "
          f"{cognee_raw.number_of_edges()} edges")
else:
    print("Skipped. Steps 4-6 will use a simulated graph so the notebook still runs.")

### Find out what cognee actually emitted, before filtering

Cognee's graph holds bookkeeping nodes — `DocumentChunk`, `TextSummary`,
`TextDocument` — alongside extracted entities. Counting those as entities would
inflate the numbers badly.

**Do not skip this cell.** The label names are not guaranteed stable across
versions, so inspect them rather than trusting a hard-coded filter. If you
filter on a label that no longer exists you get an empty graph, which reports as
"100% of entities missed" and looks like a catastrophic result instead of a
wrong filter.

In [ ]:
def summarize_node_types(G, type_attr="type"):
    counts = {}
    for _, data in G.nodes(data=True):
        key = str(data.get(type_attr, "<untyped>"))
        counts[key] = counts.get(key, 0) + 1
    return dict(sorted(counts.items(), key=lambda pair: -pair[1]))


if cognee_raw is not None:
    # The export has been seen to use 'type' or 'label'; check which is present.
    TYPE_ATTR = "type"
    if not any("type" in d for _, d in cognee_raw.nodes(data=True)):
        TYPE_ATTR = "label"

    print("type attribute:", TYPE_ATTR)
    print("node labels   :", summarize_node_types(cognee_raw, TYPE_ATTR))
    print()
    print("sample node attributes:")
    for node, data in list(cognee_raw.nodes(data=True))[:3]:
        print(" ", node, "->", dict(list(data.items())[:6]))

### Keep only the entity nodes

Set `ENTITY_LABELS` from what you just saw. `{"Entity"}` is the expected value
for cognee 1.4.x; change it if the cell above says otherwise.

In [ ]:
ENTITY_LABELS = {"Entity"}

def simulate_llm_extraction(corpus, seed=0):
    """Stand-in so the rest of the notebook runs without an API key.

    Models the documented failure mode: an extractor is **consistent within a
    document and inconsistent across documents**. Each document picks one
    surface form per entity and links what that document states; a different
    document may pick a different form for the same entity.

    Getting this shape right took three attempts, and the wrong versions are
    instructive:

    1. Orphan alias nodes with no edges. Not how extractors fail - duplicates
       are harmful precisely *because* they carry edges.
    2. Edges scattered uniformly at random across variants. This made the
       duplicates' neighbourhoods almost disjoint, which is the hardest possible
       case for structural matching and not representative either.
    3. Per-document consistency (this version). Two variants of one entity share
       neighbours to the extent their documents mention the same other entities,
       which is what actually happens.

    Still INVENTED. Never report numbers from this - it exists only to exercise
    steps 4-6 without an API key.
    """
    import random

    rand = random.Random(seed)
    G = nx.DiGraph()

    variants = {}
    for entity in corpus.entities:
        ids = [entity.id]
        G.add_node(entity.id, name=entity.name, type="Entity")
        for index, alias in enumerate(entity.aliases):
            node = f"{entity.id}__{index}"
            G.add_node(node, name=alias, type="Entity")
            ids.append(node)
        variants[entity.id] = ids

    relations_by_pair = {}
    for relation in corpus.relations:
        relations_by_pair.setdefault((relation.source, relation.target), relation.type)

    for document in corpus.documents:
        mentioned = set(document.entity_ids)
        # One choice per entity per document - the "consistent within a chunk"
        # part. Seeded by document id so the corpus stays reproducible.
        chosen = {
            entity_id: rand.choice(variants[entity_id]) for entity_id in mentioned
        }
        for (source, target), rel_type in relations_by_pair.items():
            if source in mentioned and target in mentioned:
                G.add_edge(chosen[source], chosen[target], relationship=rel_type)
    return G


if cognee_raw is not None:
    keep = [n for n, d in cognee_raw.nodes(data=True)
            if str(d.get(TYPE_ATTR, "")) in ENTITY_LABELS]
    if not keep:
        raise ValueError(
            f"No nodes matched {ENTITY_LABELS}. Set it from the labels printed above."
        )
    extracted = cognee_raw.subgraph(keep).copy()
    for _, data in extracted.nodes(data=True):
        data.setdefault("name", data.get("text", ""))
    LABEL = "cognee 1.4.1"
    IS_REAL = True
else:
    extracted = simulate_llm_extraction(corpus, seed=SEED)
    LABEL = "SIMULATED (not a real result)"
    IS_REAL = False

print(f"{LABEL}: {extracted.number_of_nodes()} entity nodes")

---
## Step 4 — Measure

`duplication_rate` is the headline: of the entities the pipeline found at all,
what share did it split across more than one node.

Read `unmatched_nodes` as a caveat on everything else. A pipeline that invents
unrelated nodes, or labels them in a way the corpus cannot recognise, shows a
high count there — and its other numbers deserve proportionally less trust.
Attribution is itself a matching problem, so unmatched nodes are **reported
rather than forced** onto the nearest entity.

In [ ]:
before = duplication_report(extracted, corpus, framework=LABEL)
print(before.summary())

if not IS_REAL:
    print("\n*** SIMULATED DATA - not a measurement of any real framework. ***")

### Look at what got split

The aggregate number is what goes in a table; these examples are what make it
believable to a reader. Check them: if the "duplicates" are actually distinct
entities, the corpus is at fault and you should revisit step 1.

In [ ]:
from graphfaker.corpus import attribute_nodes  # noqa: F401

attribution = attribute_nodes(extracted, corpus)
for entity_id, count in before.worst[:8]:
    entity = lookup[entity_id]
    nodes = attribution.matched.get(entity_id, [])
    names = [extracted.nodes[n].get("name", n) for n in nodes]
    print(f"{entity.name!r} ({entity.type}) -> {count} nodes")
    for name in names:
        print(f"     {name!r}")
    print()

if attribution.unmatched:
    print(f"{len(attribution.unmatched)} nodes matched no known entity, e.g.:")
    for node in attribution.unmatched[:5]:
        print("  ", repr(extracted.nodes[node].get("name", node)))

---
## Step 5 — Repair the graph, then re-measure

A diagnosis on its own is a complaint. `resolve()` scores candidate pairs on
attribute similarity **and** neighbourhood overlap — two nodes sharing most of
their neighbours are probably the same entity, however differently their names
are spelled. That is the signal a tabular record-linkage tool structurally
cannot use, because it has no graph.

The combination is `attr + w * structural * (1 - attr)`, so structure only ever
**raises** a score. An isolated node is never penalised for having few
neighbours.

Then it merges each cluster onto one canonical node, rewiring incident edges,
dropping self-loops the merge creates, and recording what it absorbed in
`_merged_from` so the merge is not silently lossy.

### Where this gets hard, stated plainly

Structural matching assumes duplicate nodes **share neighbours**. That holds when
two mentions of an entity appear alongside the same other entities. It fails when
an extractor *partitions* an entity's edges across its duplicates, because then
the duplicates' neighbourhoods are close to disjoint and there is no overlap to
find.

So do not expect one setting to be right. Sweep the threshold and read precision
and recall together:

- **High threshold** — merges only near-certain pairs. High precision, low recall.
- **Low threshold** — catches more real duplicates and starts merging distinct
  entities. A resolver that merges everything scores a perfect duplication rate
  and destroys the graph.

Pick the operating point from these numbers, not from the duplication rate alone.

In [ ]:
from graphfaker.resolve import evaluate_clusters

gold_clusters_probe = [n for n in attribute_nodes(extracted, corpus).matched.values()
                       if len(n) > 1]

print(f"{'threshold':>10s} {'struct_w':>9s} {'clusters':>9s} {'merged':>7s} "
      f"{'precision':>10s} {'recall':>7s} {'F1':>6s}")
sweep = []
for threshold in (0.90, 0.85, 0.82, 0.75, 0.70, 0.65):
    candidate = resolve_entities(
        extracted, on=["name"], threshold=threshold, structural_weight=0.6
    )
    s = evaluate_clusters(candidate.clusters, gold_clusters_probe)
    sweep.append((threshold, candidate, s))
    print(f"{threshold:10.2f} {0.6:9.1f} {len(candidate.clusters):9d} "
          f"{candidate.duplicate_nodes:7d} {s['pairwise_precision']:10.3f} "
          f"{s['pairwise_recall']:7.3f} {s['pairwise_f1']:6.3f}")

Now pick the setting. `THRESHOLD` below is the one carried into the rest of the
notebook — change it based on the sweep, and say in any write-up which value you
used and why.

In [ ]:
THRESHOLD = 0.70   # from the sweep above: precision still 1.000, best F1

result = resolve_entities(
    extracted,
    on=["name"],
    threshold=THRESHOLD,
    structural_weight=0.6,
)
print(result.report())

In [ ]:
repaired = result.apply()
after = duplication_report(repaired, corpus, framework=f"{LABEL} + resolve()")

rows = [before, after]
print(f"{'':38s} {'nodes':>7s} {'found':>7s} {'split':>7s} {'dup rate':>10s} {'inflation':>10s}")
for report in rows:
    print(f"{report.framework[:38]:38s} {report.nodes_in_graph:7d} {report.found:7d} "
          f"{report.duplicated:7d} {report.duplication_rate:9.1%} {report.node_inflation:9.2f}x")

print()
removed = before.nodes_in_graph - after.nodes_in_graph
if before.duplicated:
    consolidated = before.duplicated - after.duplicated
    print(f"duplicate nodes removed      : {removed}")
    print(f"entities reduced to one node : {consolidated}/{before.duplicated}")
    if removed and not consolidated:
        # Worth distinguishing: partial progress on an entity split three ways
        # still leaves it counted as duplicated.
        print("\nNodes were merged but no entity was fully consolidated - each")
        print("still has 2+ nodes. Partial merges do not move the dup rate.")
else:
    print("Nothing was split, so there was nothing to repair.")

### Did it over-merge?

Recall is easy; precision is the hard part. A resolver that merges everything
scores a perfect duplication rate and destroys the graph. `evaluate_clusters`
scores the clustering against the gold grouping, so over-merging shows up as
poor precision.

This uses labels the corpus already holds — it does not manufacture ground
truth.

In [ ]:
from graphfaker.resolve import evaluate_clusters

gold_clusters = [
    nodes for nodes in attribution.matched.values() if len(nodes) > 1
]
scores = evaluate_clusters(result.clusters, gold_clusters)

for key in ("pairwise_precision", "pairwise_recall", "pairwise_f1",
            "b_cubed_precision", "b_cubed_recall", "b_cubed_f1"):
    print(f"  {key:20s} {scores[key]:.3f}")
print()
print(f"  predicted pairs {scores['predicted_pairs']}, "
      f"gold pairs {scores['gold_pairs']}, correct {scores['correct_pairs']}")
print()
print("Low precision means resolve() merged entities that are genuinely distinct.")
print("Low recall means it left real duplicates behind. Tune `threshold` and")
print("`structural_weight` against these numbers, not against the dup rate alone.")

---
## Step 6 — Load the repaired graph into a database

File-based, so no driver and no running database is needed. The CSV path also
covers TigerGraph `LOAD`, Amazon Neptune's bulk loader, and Spark/GraphFrames.

In [ ]:
export_cypher(repaired, "repaired.cypher", dialect="neo4j")
export_neo4j_csv(repaired, "neo4j_import")
export_cypher(repaired, "repaired.gql", dialect="gql")

print(open("repaired.cypher").read()[:600])

Then, in Neo4j, graph data science runs directly on the result:

```cypher
CALL gds.graph.project('g', '*', '*');

CALL gds.pageRank.stream('g') YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS name, score
ORDER BY score DESC LIMIT 10;

CALL gds.louvain.stream('g') YIELD nodeId, communityId
RETURN communityId, count(*) AS size ORDER BY size DESC;
```

---
## Step 7 — Record the result

Save the numbers with the seed and versions attached. A result without the seed
is not reproducible, and a result without the corpus audit is not defensible.

In [ ]:
record = {
    "seed": SEED,
    "graphfaker_version": graphfaker.__version__,
    "corpus": {
        "entities": corpus.expected_entity_count,
        "documents": len(corpus.documents),
        "audit_clean": audit["clean"],
    },
    "is_real_measurement": IS_REAL,
    "results": [before.row(), after.row()],
    "resolve_quality": scores,
    "per_entity_before": before.per_entity,
}
with open("results.json", "w") as handle:
    json.dump(record, handle, indent=2)

print(json.dumps(record["results"], indent=2))
if not IS_REAL:
    print("\nis_real_measurement is false - simulated run.")

---
## Caveats to carry into any write-up

State these alongside the number. They are what separate a result from a claim.

1. **Clean input only.** Real documents are messier. This is a lower bound on
   duplication, not an estimate of production behaviour.
2. **One corpus, one seed, one run.** LLM extraction is non-deterministic —
   [langchain#26614](https://github.com/langchain-ai/langchain/issues/26614)
   documents identical input yielding a typed node on one run and an untyped one
   on the next. Run several seeds and report the spread, not one number.
3. **Attribution is itself a matching problem.** `unmatched_nodes` is the
   honest measure of how much the harness could not account for. Report it.
4. **Entity-label filtering is a judgement call.** Which node types count as
   entities is set by hand in step 3, and a reader may disagree. Say what you
   filtered.
5. **Synthetic names are not real names.** Faker's distributions are not a
   population. Cross-cultural naming, honorifics, and transliteration are all
   places real extraction fails that this corpus does not probe.
6. **This does not measure answer quality.** A pipeline can duplicate entities
   and still answer questions well. Do not extrapolate.

### Reproducing

```bash
pip install graphfaker cognee
export LLM_API_KEY=...
jupyter lab docs/notebooks/duplication_experiment.ipynb
```

Or headless, across several seeds:

```bash
for seed in 1 2 3 7 42; do
  python examples/duplication_experiment.py --seed $seed --entities 60 --documents 80 \
    --results results_$seed.json
done
```